# Regular Expressions

A regular expression (shortened as regex or regexp), sometimes referred to as a rational expression, is a sequence of characters that specifies a match pattern in text. Usually such patterns are used by string-searching algorithms for "find" or "find and replace" operations on strings, or for input validation. Regular expression techniques are developed in theoretical computer science and formal language theory.

In [25]:
import re
from typing import Self, Optional
from dataclasses import dataclass, field
import colorama

In [26]:
res = re.search("seminars?k?i?","seminar")
if res:
    print("Found:")
    print(f"Raw: {res}")
    print(f"Group: {res.group()}")
else:
    print("Not found")

Found:
Raw: <re.Match object; span=(0, 7), match='seminar'>
Group: seminar


# Types of Regex engines
There are two types of Regex engines:
1. Regex-directed engines ( NFA engines )
2. Text-directed engines  ( DFA engines )

There are pros and cons to each model, the main ones being that NFA engines support more complex patterns, but can run in O(2^n) in worst case scenarios, and that DFA is strictly linear in execution time (O(n)).

A finite automation is defined by five components:  (**Q**, **Σ**, **δ**, **q₀**, **F**): 
1. Q: Finite set of all possible states
2. Σ (Sigma): A finite set of input symbols (the alphabet).
3. δ (Delta): The transition function that maps a state and an input symbol to the next state.
4. q₀: The initial state where the process starts.
5. F: A set of final or accepting states. In our case, True / False.

In this project, we will implement a Regex-directed engine.


In [27]:
class StateContext():
    text = None
    index = 0
    captured_groups = []
    
    def __init__(self,text,index=0,captured_groups=[]):
        self.text = text
        self.index = index
        self.captured_groups = captured_groups
        pass

In [28]:
@dataclass
class Node():
    _type = "" # Literal (leaf) or Operator (applied to children)
    def __init__(self,val):
        self.val = val
        self.children = []
        
    def accept(self,visitor):
        method_name = f'visit_{self.__class__.__name__.lower()}'
        visitor_method = getattr(visitor,method_name,visitor.generic_visit())
        return visitor_method(self)
    
    def match(self, ctx : StateContext, remaining: list[Self]):
        pass

@dataclass
class Literal(Node):
    char : str
    
    def match(self, ctx: StateContext, remaining: list[Node]) -> bool:
        if ctx.index < len(ctx.text) and ctx.text[ctx.index] == self.char:
            ctx.index += 1
            if not remaining or remaining[0].match(ctx, remaining[1:]):
                return True
            ctx.index -= 1 # Backtrack
        return False

@dataclass
class Concatenation(Node):
    children : list[Node]
    
    def match(self, ctx: StateContext, remaining: list[Node] = None) -> bool:
        if not self.children:
            return True
            
        all_remaining = self.children + (remaining or [])
        
        return all_remaining[0].match(ctx, all_remaining[1:])
    
@dataclass
class Alternation(Node):
    left : Node
    right : Node
    
    def match(self, ctx: StateContext, remaining : list[Node]) -> bool:
        start_index = ctx.index
        
        if self.left.match(ctx,remaining):
            return True
        else:
            ctx.index = start_index
            return self.right.match(ctx,remaining)

@dataclass
class Wildcard(Node):
    def match(self, ctx: StateContext, remaining: list[Node]) -> bool:
        if ctx.index < len(ctx.text) and ctx.text[ctx.index] != '\n':
            ctx.index += 1 
            if not remaining or remaining[0].match(ctx, remaining[1:]):
                return True
                
            ctx.index -= 1
            
        return False

@dataclass
class Quantifier(Node):
    child : Node
    min_repeat : int
    max_repeat : Optional[int]
    
    def match(self, ctx: StateContext, remaining: list[Node]) -> bool:
        start_index = ctx.index
        match_indices = [start_index]

        while self.max_repeat is None or (len(match_indices) - 1) < self.max_repeat:
            if self.child.match(ctx, []): 
                match_indices.append(ctx.index)
            else:
                break

        while len(match_indices) - 1 >= self.min_repeat:
            ctx.index = match_indices.pop() # Try this length
            
            if not remaining:
                return True
            
            next_node = remaining[0]
            if next_node.match(ctx, remaining[1:]):
                return True
        
        ctx.index = start_index
        return False
    
@dataclass
class Group(Node):
    child : Node
    index : int
    is_capturing : bool
    
@dataclass
class End(Node):
    def match(self, ctx: StateContext, remaining: list[Node] = None) -> bool:
        return ctx.index == len(ctx.text)

In [29]:
class Tester():
    tests: dict[str, list[tuple[Node,str, bool]]]
    def __init__(self,raw_tests):
        self.tests = {}
        for rule,tree,string,expected in raw_tests:
            if rule in self.tests.keys():
                self.tests[rule].append((tree,string,expected))
            else:
                self.tests[rule] = [(tree,string,expected)]
        pass
    
    def test(self, num=-1):
        i = 0
        res = []

        for rule_name, tests in self.tests.items():
            for (tree,string,answer) in tests:
                if num==-1 or num==i:
                    if tree.match(StateContext(string)) == answer:
                        # Number, expected, Correct, String
                        res.append([i,answer,True, string,rule_name])
                    else:
                        res.append([i,not answer,False, string,rule_name])
                i+=1
        
        for r in res:
            print(f"Test {str(r[0]):<4}| "
            f"Pass: {colorama.Fore.GREEN if r[2] else colorama.Fore.RED}{'PASS' if r[2] else 'FAIL':<6}{colorama.Style.RESET_ALL}| "
            f"Rule: \"{(r[4]+f"\""):<15}|"
            f"Str: {str(r[3]):<15}| "
            f"Res: {colorama.Fore.GREEN if r[1] else colorama.Fore.RED}{str(r[1]):<6}{colorama.Style.RESET_ALL}|")

# Lexer
A lexer is an automaton, in this case, a **Nondeterministic Finite Automaton**. An automaton is an abstract mathematical model of a computing machine. It only knows how to read a sequence of inputs, and move between different states based on pre defined rules.

A standard Finite State Automaton (FSA) consists of:
1. States - Specific states the machine can be in.
2. Transitions - The rules dictating how the machine moves from one state  to another.
3. An alphabet - The set of valid input symbols. In our case, the alphabet + some special characters.
4. A start state - Where the machine begins.
5. Accepting states - One or more final states.

---

# Determinism vs. Nondeterminism
1. DFA (Deterministic Finite Automation) - For any given state and any given input, there is excactly one possible next state.
2. NFA (Nondeterministic Finite Automation) - For a given state and input, there can be multiple valid next states, which must be explored until a working one is found.

In [30]:
@dataclass
class Token:
    type: str  
    value: str 

class Lexer:
    def __init__(self, text: str):
        self.text = text

    def tokenize(self) -> list[Token]:
        tokens = []
        i = 0
        while i < len(self.text):
            char = self.text[i]
            
            if char == '*':
                tokens.append(Token('STAR', char))
            elif char == "?":
                tokens.append(Token('QMARK',char))
            elif char == "+":
                tokens.append(Token("PLUS",char))
            elif char == '.':
                tokens.append(Token("DOT", char))
            elif char == '|':
                tokens.append(Token("PIPE",char))
            elif char == '(':
                tokens.append(Token("LPAREN",char))
            elif char == ')':
                tokens.append(Token("RPAREN",char))
            elif char == '\\':
                if i + 1 < len(self.text):
                    tokens.append(Token('CHAR', self.text[i+1]))
                    i += 1
                else:
                    raise ValueError("Unused escape char at end of string")
            else:
                tokens.append(Token('CHAR', char))
            i += 1
        return tokens

# Parsing Regex

In regex, the precendence from highest to lowest is:
1. Parentheses / Base units: literals, wildcards, etc.
2. Quantifiers: **\***, **?**, **+**
3. Concatenation: **abc**
4. Alternation **a|b**

To implement this, we work our way up from the bottom.
We start with ```parse_expression```, which handles the ```OR``` ( **|** ). It then asks ```parse_term``` and ```parse_expression``` to recursively evaluate the left and right side, respectively.

We then go to ```parse_term```, which then loops and keeps handling chunks until it hits something that breaks a chain ( like the end of the string, or a pipe).

The next layer down is ```parse_factor```, which grabs the current node, and then peeks to see if there is a quantifier behind it. If there is one, it wraps the atom in the corresponding quantifier and continues down.

The final layer is ```parse_atom```, which handles the literals, wild-cards, and sub-expressions, in which case it goes back to the top, with ```parse_expression```.

In [31]:
class Parser:
    def __init__(self, tokens: list[Token]):
        self.tokens = tokens
        self.pos = 0

    def peek(self) -> Token | None:
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return None
    
    def consume(self, expected_type : str) -> Token | None:
        token = self.peek()
        if not token or token.type != expected_type:
            raise ValueError(f"Expected {expected_type}, got {token.type if token else None}")
        self.pos +=1
        return token

    def parse_atom(self) -> Node:
        token = self.peek()
        if not token:
            raise ValueError("Unexpected end of input")
    
        if token.type == "CHAR":
            self.consume("CHAR")
            return Literal(token.value)

        elif token.type == "DOT":
            self.consume("DOT")
            return Wildcard()
        
        
        elif token.type == "LPAREN":
            self.consume("LPAREN")
            node = self.parse_expression()
            self.consume("RPAREN")
            return node
        
        else:
            raise ValueError(f"Unexpected token @ index: {self.pos} : {token.type}")
            
    def parse_factor(self) -> Node:
        node = self.parse_atom()
        
        token = self.peek()
        if token:
            if token.type == "STAR":
                self.consume("STAR")
                return Quantifier(child=node,min_repeat=0,max_repeat=None)

            elif token.type == "QMARK":
                self.consume("QMARK")
                return Quantifier(child=node,min_repeat=0,max_repeat=1)
            
            elif token.type == "PLUS":
                self.consume("PLUS")
                return Quantifier(child=node,min_repeat=1,max_repeat=None)
        return node

    def parse_expression(self) -> Node:
        left = self.parse_term()
        
        token = self.peek()
        if token and token.type == "PIPE":
            self.consume("PIPE")
            right = self.parse_expression()
            return Alternation(left,right)

        return left
    
    def parse_term(self) -> Node:
        nodes = []
        while self.peek() and self.peek().type not in ["PIPE", "RPAREN"]:
            nodes.append(self.parse_factor())
        
        if not nodes:
            return Concatenation([])

        if len(nodes) == 1:
            return nodes[0]
        
        return Concatenation(nodes)
        

    def parse(self) -> Node:
        root_node = self.parse_expression()
        return Concatenation([root_node,End()])

In [32]:
lexer_1 = Lexer(r"ab.d_*\.J?")
tokens_1 = lexer_1.tokenize()
parser_1 = Parser(tokens_1)
root_1 = parser_1.parse()

lexer_2 = Lexer(r"a*a")
tokens_2 = lexer_2.tokenize()
parser_2 = Parser(tokens_2)
root_2 = parser_2.parse()

lexer_3 = Lexer(r"a*b")
tokens_3 = lexer_3.tokenize()
parser_3 = Parser(tokens_3)
root_3 = parser_3.parse()

lexer_4 = Lexer(r"a...b")
tokens_4 = lexer_4.tokenize()
parser_4 = Parser(tokens_4)
root_4 = parser_4.parse()

lexer_5 = Lexer(r"a+b")
tokens_5 = lexer_5.tokenize()
parser_5 = Parser(tokens_5)
root_5 = parser_5.parse()

lexer_6 = Lexer(r"cat|dog")
tokens_6 = lexer_6.tokenize()
parser_6 = Parser(tokens_6)
root_6 = parser_6.parse()

tests2 = [
  # misc tests
  [r"ab.d_*\.J?",root_1,"abcd___.J",True],
  [r"ab.d_*\.J?",root_1,"abcd_.J",True],
  [r"ab.d_*\.J?",root_1,"ab--d.J",False],
  [r"ab.d_*\.J?",root_1,"abcd___.",True],
  [r"ab.d_*\.J?",root_1,"ab_x_d___.J",False],
  [r"ab.d_*\.J?",root_1,"abcd.J",True],
  [r"ab.d_*\.J?",root_1,"abxd.",True],
  # a*a tests:
  [r"a*a",root_2,"aa",True],
  [r"a*a",root_2,"aaa",True],
  [r"a*a",root_2,"a",True],
  [r"a*a",root_2,"aba",False],
  [r"a*a",root_2,"abbb",False],
  # a*b tests:
  [r"a*b",root_3,"ab",True],
  [r"a*b",root_3,"abb",False],
  [r"a*b",root_3,"b",True],
  [r"a*b",root_3,"bb",False],
  # a...b tests:
  [r"a...b",root_4,"a_x_b",True],
  [r"a...b",root_4,"a__b",False],
  [r"a...b",root_4,"a_.x._b",False],
  [r"a...b",root_4,"aaxbb",True],
  # a+b tests:
  [r"a+b",root_5,"ab",True],
  [r"a+b",root_5,"b",False],
  [r"a+b",root_5,"aaaab",True],
  # cat | dog tests
  [r"cat|dog",root_6,"cat",True],
  [r"cat|dog",root_6,"dog",True],
  [r"cat|dog",root_6,"ca",False],
  [r"cat|dog",root_6,"do",False],
  [r"cat|dog",root_6,"catdog",False],
  [r"dog|cat",root_6,"dog|c",False],
  [r"dog|cat",root_6,"d|cat",False],
]

tester = Tester(tests2)
tester.test()

Test 0   | Pass: PASS  | Rule: "ab.d_*\.J?"    |Str: abcd___.J      | Res: True  |
Test 1   | Pass: PASS  | Rule: "ab.d_*\.J?"    |Str: abcd_.J        | Res: True  |
Test 2   | Pass: PASS  | Rule: "ab.d_*\.J?"    |Str: ab--d.J        | Res: False |
Test 3   | Pass: PASS  | Rule: "ab.d_*\.J?"    |Str: abcd___.       | Res: True  |
Test 4   | Pass: PASS  | Rule: "ab.d_*\.J?"    |Str: ab_x_d___.J    | Res: False |
Test 5   | Pass: PASS  | Rule: "ab.d_*\.J?"    |Str: abcd.J         | Res: True  |
Test 6   | Pass: PASS  | Rule: "ab.d_*\.J?"    |Str: abxd.          | Res: True  |
Test 7   | Pass: PASS  | Rule: "a*a"           |Str: aa             | Res: True  |
Test 8   | Pass: PASS  | Rule: "a*a"           |Str: aaa            | Res: True  |
Test 9   | Pass: PASS  | Rule: "a*a"           |Str: a              | Res: True  |
Test 10  | Pass: PASS  | Rule: "a*a"           |Str: aba            | Res: False |
Test 11  | Pass: PASS  | Rule: "a*a"           |Str: abbb           | Res: False |
Test